# 05 — GPU vs CPU benchmark (G5)

Formal same-host benchmark: pinned CPU threads, warmup excluded, CUDA event timing on GPU, separate compute-only and end-to-end (incl. host→device copy) numbers. The CPU side takes a while — that is the point.

**Storage contract** — code lives on GitHub, data & checkpoints live on Drive (`MyDrive/s2cd`), `/content` is disposable scratch. Every notebook is self-contained: run all cells top-to-bottom in a fresh GPU session.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
import os

REPO_URL = "https://github.com/Aadith-Geeth-Mohan/sentinel2-change-detection.git"
REPO = "/content/s2-change-detection"
BRANCH = "chennai"          # the whole 00-07 series runs this branch
if os.path.exists(REPO):
    !git -C {REPO} checkout {BRANCH}
    !git -C {REPO} pull --ff-only origin {BRANCH}
else:
    !git clone -b {BRANCH} {REPO_URL} {REPO}
%cd {REPO}
!pip install -q .
import sys
sys.path.insert(0, os.path.join(REPO, "src"))
# drop cached s2cd modules so a just-pulled fix takes effect in THIS kernel
for k in [k for k in list(sys.modules) if k == "s2cd" or k.startswith("s2cd.")]:
    del sys.modules[k]


In [ ]:
import torch
from s2cd.data import DriveStore
from s2cd.utils import get_device, describe_device, load_config

device = get_device()
print(describe_device(device))
store = DriveStore()  # Drive root: MyDrive/s2cd · scratch: /content


In [ ]:
from s2cd.gpu.benchmark import run
from s2cd.train import build_model

cfg = load_config("configs/train.yaml")
model = build_model(cfg)
results = run(model, batch=16, channels=cfg["in_channels"], chip=256,
              iters=20, warmup=10)


In [ ]:
import json

rep = store.scratch / "reports"; rep.mkdir(exist_ok=True)
(rep / "benchmark.json").write_text(json.dumps(
    {**results, "device": describe_device(device)}, indent=1, default=str))
store.push_report("benchmark.json")
print("pushed benchmark.json -> Drive")


In [ ]:
# The headline chart: same host, same batch - CPU (all cores) vs A100 GPU.
import matplotlib.pyplot as plt

gpu, cpu = results["cuda"], results["cpu"]
batch = gpu["samples_per_sec"] * gpu["end_to_end_sec"]  # images per timed pass

cpu_sps = [batch / cpu["compute_sec"], cpu["samples_per_sec"]]
gpu_sps = [batch / gpu["compute_sec"], gpu["samples_per_sec"]]
speedups = [g / c for c, g in zip(cpu_sps, gpu_sps)]
print(f"GPU speedup: {speedups[0]:.1f}x (compute only), "
      f"{speedups[1]:.1f}x (end-to-end incl. copy to GPU)")

x = [0, 1]
width = 0.35
fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.bar([i - width / 2 for i in x], cpu_sps, width, color="C1",
       label=f"CPU ({cpu['threads']} threads)")
ax.bar([i + width / 2 for i in x], gpu_sps, width, color="C0",
       label="GPU (fp16 AMP)")
for i, (c, g, s) in enumerate(zip(cpu_sps, gpu_sps, speedups)):
    ax.text(i - width / 2, c, f"{c:.0f}", ha="center", va="bottom", fontsize=9)
    ax.text(i + width / 2, g, f"{g:.0f}", ha="center", va="bottom", fontsize=9)
    ax.text(i, max(c, g) * 1.5, f"{s:.0f}x faster", ha="center",
            va="bottom", fontsize=11)
ax.set_yscale("log")
ax.set_ylim(top=max(max(cpu_sps), max(gpu_sps)) * 8)
ax.set_xticks(x)
ax.set_xticklabels(["model forward pass\n(compute only)",
                    "full pass\n(incl. copy to GPU)"])
ax.set_ylabel("images processed per second (log scale)")
ax.set_title(f"GPU vs CPU - same host, batch {batch:.0f}")
ax.legend()
plt.tight_layout()
rep = store.scratch / "reports"; rep.mkdir(exist_ok=True)
fig.savefig(rep / "benchmark.png", dpi=150, bbox_inches="tight")
store.push_report("benchmark.png")
print("pushed benchmark.png -> Drive")
plt.show()
